# Test

In [ ]:
import os
import re
import glob
import anndata as ad

from deepspatial import DeepSpatial

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
# 2. 数据准备与路径解析
data_dir = "/data/yuhangyang/STereo/data/merfish" 
file_paths = sorted(glob.glob(os.path.join(data_dir, "*.h5ad")), 
                    key=lambda x: int(re.search(r'merfish_(\d+)', x).group(1)))

slice_gap = 10.0  # 物理间距 (um)

# 加载数据并注入物理 Z 坐标
adata_list = []
for p in file_paths:
    adata = ad.read_h5ad(p)
    # 解析文件名中的序号，计算物理高度并存入 adata.obs (符合新版 API 逻辑)
    idx = int(re.search(r'merfish_(\d+)', p).group(1))
    adata.obs['z_coord'] = float(idx * slice_gap) 
    adata_list.append(adata)

print(f"Loaded {len(adata_list)} slices with physical Z-coordinates in adata.obs['z_coord']")

In [ ]:
adata_list

In [ ]:
model = DeepSpatial()

# 4. 数据配置 (内部自动完成归一化，不再污染原始坐标)
model.setup_data(
    adata_list=adata_list, 
    spatial_key='spatial',   # 原始坐标键
    z_key='z_coord',         # 物理高度键
    label_key='cell_class',  # 标签键 (已从 cell_label_key 统一)
    batch_size=2048
)

In [ ]:
# 5. 构建模型 (架构参数在此处注入)
model.build_model(
    patch_size=8,
    hidden_size=256,
    depth=6,
    lr=2e-4
)

In [ ]:
# 6. 训练 (指定保存目录，设备自动识别)
model.fit(
    max_epochs=1, 
    save_dir="./checkpoints/merfish_run1",
    accelerator='gpu', 
    devices=[0],
    save_ckpt=True
)


In [ ]:
adata_3d = ad.read_h5ad('output/3d_test.h5ad')

In [ ]:
adata_3d = model.reconstruct_full_volume(
    adata_list,
    thickness=10
)

In [ ]:
adata_3d = ad.read_h5ad("output/3d_test.h5ad")

In [ ]:
interactive_3d_labels(adata_3d, color_col='Region', width=1000, height=1000)

In [ ]:
interactive_3d_expression(adata_3d, gene_name='Ucn3',  width=1000, height=1000)

In [ ]:
interactive_spatial_range_widget(adata_3d, 'Region', bg_color='black')

In [ ]:
plot_3d_labels(adata_3d, z_stretch=1, color_col='Region')

In [ ]:
plot_orthogonal_projections(adata_3d, color_col='Region')

In [ ]:
plot_z_distribution(adata_3d, color_col='Region', smooth_sigma=2)

In [ ]:
plot_virtual_slice(adata_3d, 'transverse', thickness=30, point_size=5, color_col='Region')